In [1]:
import argparse
import json
import os
from pathlib import Path

import numpy as np
import mediapy

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from common import (  # noqa: E402
    cumulative_episode_bounds,
    episode_shard_path,
    load_episode_frame,
    load_episode_records,
    load_json,
    load_lerobot_dataset,
    save_json_atomic,
)

skill_annotations = "../../pace/openpi/data/libero-100/skill_target_traces.json"

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /work/11430/jpeng303/vista/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


In [2]:
def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_depths": True}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

class LiberoEnvMaker:
    def __init__(self, suite: str,
                 render_resolution: int = 256, seed: int = 0,
                 repeats: int = 1):
        benchmark_dict = benchmark.get_benchmark_dict()
        self.task_suite = benchmark_dict[suite]()
        self.repeats = repeats
        self.render_resolution = render_resolution
        self.seed = seed

    def get_num_tasks(self):
        return self.task_suite.n_tasks

    def task_instantiations(self, task_id):
        task = self.task_suite.get_task(task_id)
        initial_states = self.task_suite.get_task_init_states(task_id)
        env, task_description = _get_libero_env(task, self.render_resolution, self.seed)
        for episode_idx in range(self.repeats):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])
            yield obs, env, task_description


In [3]:
with open(skill_annotations) as f:
    annotation_data = json.load(f)
repo_id = annotation_data['source_repo_id']
dataset_root = os.path.dirname(skill_annotations)

records = load_episode_records(dataset_root)
episode_bounds = cumulative_episode_bounds(records)
print("loading lerobot dataset...", flush=True)
dataset = load_lerobot_dataset(repo_id, dataset_root)
print("dataset loaded", flush=True)

loading lerobot dataset...


The dataset you requested (yilin-wu/libero-100) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=yilin-wu/libero-100
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

dataset loaded


In [10]:
N_REPEATS = 9999
libero_90 = LiberoEnvMaker("libero_90", repeats=N_REPEATS)
libero_10 = LiberoEnvMaker("libero_10", repeats=N_REPEATS)
task_generators = [libero_90.task_instantiations(i) for i in range(libero_90.get_num_tasks())] \
                + [libero_10.task_instantiations(i) for i in range(libero_10.get_num_tasks())]

# for counter, (episode_index, (start, end)) in enumerate(episode_bounds.items()):
#     annot_data = annotation_data[str(episode_index)]
#     obs, env, task_description = next(task_generators[counter % len(task_generators)])
#     video_frames = []
#     trace = annot_data['target_traces'][0]['end_effector_trace']['source_world_positions']
#     zeroed = False
#     for i in range(60):
#         if not zeroed:
#             zeros = (np.array(trace[i]), obs['robot0_eef_pos'])
#             zeroed = True
#         print(trace[i] - zeros[0], obs['robot0_eef_pos'] - zeros[1])
#         row = dataset.hf_dataset[start + i]
#         action = np.array(row['actions'])
#         obs, reward, done, info = env.step(action)
#         print(obs)
#         break
#         video_frames.append(np.copy(obs['agentview_image'][::-1, ::-1, :]))
#     mediapy.write_video("out.mp4", video_frames)
#     break

[0. 0. 0.] [0. 0. 0.]
OrderedDict({'robot0_joint_pos': array([-0.01672111, -0.16644746, -0.01725659, -2.43433574, -0.01797804,
        2.20768833,  0.83079721]), 'robot0_joint_pos_cos': array([ 0.99986021,  0.98617957,  0.99985111, -0.76014706,  0.9998384 ,
       -0.59469965,  0.67428726]), 'robot0_joint_pos_sin': array([-0.01672033, -0.16567996, -0.01725573, -0.64975107, -0.01797707,
        0.80394796,  0.73846916]), 'robot0_joint_vel': array([ 0.0500847 ,  0.03838912,  0.124143  , -0.02575619,  0.12374677,
        0.06130282, -0.21986291]), 'robot0_eef_pos': array([-0.20833112, -0.0172375 ,  1.17864937]), 'robot0_eef_quat': array([ 0.99895194, -0.0340097 , -0.03006301,  0.0058797 ]), 'robot0_gripper_qpos': array([ 0.02111653, -0.02104852]), 'robot0_gripper_qvel': array([ 0.01929241, -0.01763897]), 'agentview_image': array([[[200, 182, 162],
        [199, 181, 161],
        [199, 181, 161],
        ...,
        [197, 180, 164],
        [194, 177, 161],
        [198, 181, 165]],

   

/tmp/ipykernel_804820/2622930962.py:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  action = np.array(row['actions'])


StopIteration: 